# Comparative evaluation: QAOA (Nexus/Helios) vs. classical baselines

This notebook benchmarks **QAOA** (run on the Quantinuum Nexus **Helios-1E-lite**
emulator) against the classical Max-Cut baselines (**brute force**, **greedy**,
**Goemans-Williamson**) on the *same* fault-zone QUBO, across grids of growing
size (9 -> 15 -> 26 nodes) and a configurable `shots` x `max_iter` sweep, all run
in parallel.

Every optimizer is scored on the full QUBO objective. Classical baselines run on
the augmented Ising graph (maximizing its cut == minimizing `<H_C>`). Brute force
enumerates the whole spectrum once per grid (timeout-guarded; on timeout GW is the
baseline). QAOA runs COBYLA on Helios, one cloud job per iteration, and the
per-shot distributions are decoded from the Nexus job results.

Because QAOA is stochastic, every configuration is repeated over `N_RUNS`
independent replicate runs so the reported mean/std are statistically meaningful
(a single run cannot give a real std). All heavy lifting lives in `src.benchmark`;
this notebook is orchestration + plots.

**Requirements:** the QAOA section needs network access and a Nexus login. The
classical + brute-force sections run fully offline.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

from src import benchmark as bm


## 2. Experiment configuration

Edit the two sweep lists and the grid/p-value sets here. `SHOTS_LIST` and
`MAXITER_LIST` are processed as a **full cross-product** (O(N*M)); start small and
scale up. `N_RUNS` is the number of independent replicate runs per config -- QAOA
needs `N_RUNS > 1` for a meaningful mean/std. Classical greedy/GW use `n_shots`
samples per run and ignore `max_iter`.

In [ ]:
bm.GRAPH_SIZES   = [9, 15, 26]   # grow from the 9-node Guanacaste-North baseline
bm.P_VALUES      = [1, 3, 6]      # QAOA layers
bm.SHOTS_LIST    = [5000]         # shots per circuit / classical samples
bm.MAXITER_LIST  = [100]          # COBYLA iterations (QAOA only)
bm.N_RUNS        = 3              # replicate runs per config (mean/std across runs)
bm.DEVICE        = "Helios-1E-lite"
bm.SEED          = 7
bm.BRUTE_FORCE_TIMEOUT_S = 300.0  # always-on; on timeout GW becomes the baseline
bm.MAX_WORKERS   = 8

RUN_QAOA = True   # set False for an offline (classical-only) dry run

print("sweep:", bm.SHOTS_LIST, "x", bm.MAXITER_LIST,
      "| sizes", bm.GRAPH_SIZES, "| p", bm.P_VALUES, "| runs", bm.N_RUNS)

## 3. Build the grids (9, 15, 26)

Each grid grows from the 9-node baseline by BFS-adding adjacent substations, then
is turned into a `CostHamiltonian`.

In [ ]:
hamiltonians = bm.grow_cost_hamiltonians(bm.GRAPH_SIZES)
for size, ch in hamiltonians.items():
    print(f"size {size:2d}: {ch.n_qubits} qubits, "
          f"{len(ch.z_terms)} single-Z, {len(ch.zz_terms)} ZZ terms")

## 4. Approximation-ratio baselines (parallel brute force)

One baseline per grid size. Brute force enumerates the full spectrum
(vectorized, timeout-guarded); if a grid times out it falls back to a GW-derived
baseline. Runs the grids in parallel.

In [ ]:
baselines = bm.compute_baselines(hamiltonians)
for size, b in sorted(baselines.items()):
    print(f"size {size:2d}: E_min={b.e_min:.4f}  E_max={b.e_max:.4f}  "
          f"source={b.source}  timed_out={b.timed_out}  ({b.time_s:.1f}s)")

## 5. Authenticate with Nexus (QAOA only)

Opens a browser to log in (token lasts ~30 days). Skip if already logged in or if
`RUN_QAOA = False`.

In [ ]:
if RUN_QAOA:
    from src import qaoa_nexus as qn
    qn.login()
    qn.get_project(bm.PROJECT)
    print("Nexus project:", bm.PROJECT)
else:
    print("RUN_QAOA is False - skipping Nexus login")

## 6. Run everything in parallel and await all tasks

Builds the task list -- classical greedy/GW per `(size, shots, run)`; QAOA per
`(size, p, shots, max_iter, run)` -- and runs them concurrently. `run_all` blocks
until **every** task finishes before returning, so nothing below runs on partial
data.

Heads-up on volume: the QAOA path submits
`sizes * p * shots * maxiter * N_RUNS` optimizations, each ~`max_iter` Helios jobs.
Do a small smoke run first.

In [ ]:
tasks = bm.build_tasks(
    hamiltonians,
    p_values=bm.P_VALUES,
    shots_list=bm.SHOTS_LIST,
    maxiter_list=bm.MAXITER_LIST,
    n_runs=bm.N_RUNS,
    include_qaoa=RUN_QAOA,
)
print(f"{len(tasks)} tasks scheduled")

records = bm.run_all(tasks, max_workers=bm.MAX_WORKERS)
print(f"{len(records)} records completed")

## 7. Metrics\n\n### 7a. Per-run table\n\nOne row per replicate run: mean/std of the sampled energies, MSE vs. the baseline optimum, and mean/best approximation ratio.

In [ ]:
rows = bm.summarize(records, baselines)
rows.sort(key=lambda r: (r["size"], r["n_shots"], str(r["max_iter"]),
                         r["optimizer"], r["run"]))
hdr = ["optimizer","size","shots","max_iter","p","run",
       "mean_E","std_E","MSE","mean_AR","best_AR","time_s"]
def fmt(v):
    if v is None: return f"{'n/a':>10}"
    if isinstance(v, float): return f"{v:>10.4f}"
    return f"{str(v):>10}"
print("  ".join(f"{h:>10}" for h in hdr))
for r in rows:
    print("  ".join(fmt(r[k]) for k in
          ["optimizer","size","n_shots","max_iter","p","run",
           "mean_energy","std_energy","mse_vs_optimum",
           "mean_approx_ratio","best_approx_ratio","time_s"]))

### 7b. Run-aggregated table (mean +/- std across runs)

The requested statistics: across the `N_RUNS` replicates, the **mean and std** of
the execution time, the best energy found, the MSE vs. the optimum, and the
approximation ratio. This is where the multiple QAOA runs pay off.

In [ ]:
agg = bm.aggregate_runs(records, baselines)
hdr = ["optimizer","size","shots","max_iter","p","n_runs",
       "time(mean+/-std)","AR(mean+/-std)","bestAR(mean+/-std)","MSE(mean+/-std)"]
print("  ".join(f"{h:>20}" for h in hdr))
def pm(m, s):
    if m is None: return f"{'n/a':>20}"
    return f"{m:.4f}+/-{(s or 0):.4f}".rjust(20)
for r in agg:
    print(f"{r['optimizer']:>20}{r['size']:>6}{r['n_shots']:>7}"
          f"{str(r['max_iter']):>9}{str(r['p']):>4}{r['n_runs']:>7}  "
          + pm(r['mean_time_s'], r['std_time_s'])
          + pm(r['mean_approx_ratio'], r['std_approx_ratio'])
          + pm(r['mean_best_approx_ratio'], r['std_best_approx_ratio'])
          + pm(r['mean_mse_vs_optimum'], r['std_mse_vs_optimum']))

## 8. Persist raw results\n\nWrites the records, baselines, per-run and run-aggregated tables to a timestamped JSON under `experiments/results/`.

In [ ]:
out = bm.save_results(records, baselines, rows, aggregated=agg)
print("saved:", out)

## 9. Figures

All figures **pool the replicate runs** of a configuration (except the convergence
plot, which shows each run so the run-to-run spread is visible).

### 9a. Energy distribution / histogram per optimizer

In [ ]:
def _pool_by_config(records):
    pooled = {}
    for r in records:
        key = (r.optimizer, r.size, r.n_shots, r.max_iter, r.p)
        pooled.setdefault(key, []).extend(r.sample_energies)
    return pooled

def plot_energy_histograms(records, baselines, sizes=None):
    sizes = sizes or sorted({r.size for r in records})
    pooled = _pool_by_config(records)
    for size in sizes:
        items = [(k, v) for k, v in pooled.items() if k[1] == size and v]
        if not items:
            continue
        base = baselines.get(size)
        all_e = np.concatenate([np.asarray(v) for _, v in items])
        lo, hi = all_e.min(), all_e.max()
        if base:
            lo, hi = min(lo, base.e_min), max(hi, base.e_max)
        bins = np.linspace(lo, hi, 41)
        fig, ax = plt.subplots(figsize=(11, 5))
        for (opt, _, shots, mit, _), energies in sorted(items):
            lbl = f"{opt} (s={shots}" + (f", m={mit}" if mit else "") + ")"
            ax.hist(energies, bins=bins, density=True, alpha=0.45, label=lbl)
        if base:
            ax.axvline(base.e_min, color="k", ls="--", lw=2,
                       label=f"optimum {base.e_min:.2f} ({base.source})")
        ax.set_title(f"Energy distribution per optimizer - {size}-node grid "
                     f"(pooled over runs)")
        ax.set_xlabel("QUBO energy (lower = better)"); ax.set_ylabel("density")
        ax.legend(fontsize=8, framealpha=0.9); ax.grid(True, ls=":", alpha=0.35)
        fig.tight_layout(); plt.show()

plot_energy_histograms(records, baselines)

### 9b. Approximation-ratio box plots

One figure per *combination* `(size, shots, max_iter)`. Boxes are the optimizers
(greedy, GW, and QAOA p=1/3/6 - all p's belong to the same combination figure),
pooling the per-shot approximation ratios across all replicate runs. The
brute-force optimum sits at ratio 1.0 (dashed line).

In [ ]:
def plot_approx_ratio_boxes(records, baselines):
    maxiters = sorted({r.max_iter for r in records if r.max_iter is not None}) or [None]
    combos = sorted({(r.size, r.n_shots) for r in records})
    for size, shots in combos:
        for mit in maxiters:
            base = baselines.get(size)
            if base is None:
                continue
            # Pool ratios across runs, grouped by optimizer.
            by_opt = {}
            for r in records:
                if r.size != size or r.n_shots != shots:
                    continue
                if r.p is not None and r.max_iter != mit:
                    continue
                by_opt.setdefault(r.optimizer, []).extend(
                    bm.approximation_ratios(r, base))
            if not by_opt:
                continue
            labels = sorted(by_opt)
            boxes = [by_opt[k] for k in labels]
            fig, ax = plt.subplots(figsize=(1.6 * len(boxes) + 3, 5))
            ax.boxplot(boxes, showmeans=True)
            ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)
            ax.axhline(1.0, color="k", ls="--", lw=1, label="optimum")
            ax.set_title(f"Approx. ratio - {size}-node grid, shots={shots}, "
                         f"max_iter={mit} (pooled over runs)")
            ax.set_ylabel("approximation ratio (1 = optimum)")
            ax.grid(True, axis="y", ls=":", alpha=0.35)
            plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
            fig.tight_layout(); plt.show()

plot_approx_ratio_boxes(records, baselines)

### 9c. Approximation-ratio convergence

For each QAOA run, the per-iteration expectation `<H_C>` (decoded from the Nexus
job results) is converted to an approximation ratio and plotted vs. iteration -
one line per replicate run so the run-to-run spread is visible. Greedy/GW
cumulative-best ratios (vs. sample index, one representative run) are overlaid.

In [ ]:
def plot_convergence(records, baselines, sizes=None):
    sizes = sizes or sorted({r.size for r in records})
    for size in sizes:
        base = baselines.get(size)
        if base is None:
            continue
        fig, ax = plt.subplots(figsize=(11, 5))
        plotted = False
        seen_classical = set()
        for r in sorted(records, key=lambda r: (r.optimizer, r.run)):
            if r.size != size:
                continue
            if r.p is not None:  # QAOA: per-iteration expectation, one line per run
                hist = r.extra.get("history") or []
                if not hist:
                    continue
                ar = [bm.qmod_qaoa.approximation_ratio(e, base.e_min, base.e_max)
                      for e in hist]
                ax.plot(range(1, len(ar) + 1), ar, marker="o", ms=3, alpha=0.8,
                        label=f"{r.optimizer} m={r.max_iter} run{r.run}")
                plotted = True
            else:  # classical: cumulative-best, one representative run per optimizer
                if r.optimizer in seen_classical:
                    continue
                seen_classical.add(r.optimizer)
                e = np.asarray(r.sample_energies)
                cbest = np.minimum.accumulate(e)
                ar = [bm.qmod_qaoa.approximation_ratio(x, base.e_min, base.e_max)
                      for x in cbest]
                ax.plot(range(1, len(ar) + 1), ar, ls="--", alpha=0.7,
                        label=f"{r.optimizer} best-so-far")
                plotted = True
        if not plotted:
            plt.close(fig); continue
        ax.axhline(1.0, color="k", ls=":", lw=1)
        ax.set_title(f"Approximation-ratio convergence - {size}-node grid")
        ax.set_xlabel("iteration / sample"); ax.set_ylabel("approximation ratio")
        ax.legend(fontsize=8, framealpha=0.9); ax.grid(True, ls=":", alpha=0.35)
        fig.tight_layout(); plt.show()

plot_convergence(records, baselines)

## 10. Comparison against the brute-force baseline

For each QAOA record, compare the most-likely partition's energy to the exact
optimum (or the GW-derived baseline when brute force timed out).

In [ ]:
for r in sorted(records, key=lambda r: (r.size, r.optimizer, r.run)):
    if r.p is None:
        continue
    base = baselines.get(r.size)
    best_e = r.extra.get("best_energy")
    if base is None or best_e is None:
        continue
    gap = 100 * (best_e - base.e_min) / abs(base.e_min) if base.e_min else float("nan")
    match = abs(best_e - base.e_min) <= 1e-6
    print(f"{r.optimizer:9s} g{r.size:<2d} m={r.max_iter} s={r.n_shots} run{r.run}: "
          f"QAOA best E={best_e:.4f}  optimum={base.e_min:.4f} ({base.source})  "
          f"gap={gap:+.2f}%  match={match}")